# ExhibitFlow GPU Worker — Kaggle Experimental Notebook (T4 x2)

> **IMPORTANT — Experimental / Demo Only (NOT equivalent to Vast.ai):**
> This notebook is intended strictly for short prototyping/demo runs on Kaggle and is **not** a production replacement for Vast.ai.

### Official Kaggle Hardware & Quota Limits
- **GPU:** Free **NVIDIA T4 x2** (16 GB × 2 VRAM, ~29 GB system RAM, 4 CPU cores; P100 is retired).
- **Session & Idle Timeouts:** Max **12 hours** per session, **20-minute interactive idle timeout**, and **~30 hours/week** GPU quota.
- **Ephemeral Storage:** `/kaggle/working` is capped at **20 GB** and is wiped when the session ends (only `/kaggle/input` datasets persist across sessions).
- **Terms of Service & Networking:** Kaggle ToS restricts usage to personal, non-commercial research/prototyping (not production APIs). Cloudflare Quick Tunnel URLs (`*.trycloudflare.com`) are ephemeral and change on every restart.

### Recommended Usage
1. Enable **Accelerator: GPU T4 x2** and **Internet: On** in Kaggle Notebook Settings.
2. Attach your ExhibitFlow weights dataset under `/kaggle/input/` (optional — avoids re-downloading multi-GB weights every session) and add `HF_TOKEN` to **Kaggle Secrets** if using a private HuggingFace repo.
3. Run cells 1 -> 4 in order. Copy the printed `Public HTTPS` URL and `X-GPU-Token` into Backend / Web Settings.

In [ ]:
# 1. Environment & Pre-flight Check (GPU, CUDA, glibc, Disk, Internet)
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

import torch

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")
else:
    print("  WARNING: No CUDA GPU detected. Enable GPU T4 x2 in Kaggle Settings or use DEMO_MODE = True.")

subprocess.run(["nvidia-smi"], check=False)

ldd_out = subprocess.check_output(["ldd", "--version"], text=True).splitlines()[0]
print("glibc:", ldd_out)

work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
du = shutil.disk_usage(work_dir)
print(f"Disk ({work_dir}): {du.free / (1024**3):.1f} GB free / {du.total / (1024**3):.1f} GB total")

try:
    with urllib.request.urlopen("https://huggingface.co", timeout=8) as r:
        print("Internet check: OK (HTTP", r.status, ")")
except Exception as exc:
    print("Internet check: FAILED ->", exc, "(Enable Internet = On in Kaggle Settings)")

In [ ]:
# 2. Clone/Update ExhibitFlow & Configure Smart Cache (/kaggle/input + Kaggle Secrets)
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/khangkaka066/ExhibitFlow.git"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/ExhibitFlow")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-f", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

GPU_DIR = REPO_DIR / "GPU_server"
os.chdir(GPU_DIR)

# Load HF_TOKEN safely from Kaggle Secrets if configured (never hardcode secrets)
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
        print("Loaded HF_TOKEN from Kaggle Secrets.")
    except Exception:
        print("HF_TOKEN not found in Kaggle Secrets (ok for public repos or DEMO_MODE).")

# Smart Cache: scan /kaggle/input for pre-cached ExhibitFlow weights & HF/GGUF cache
WEIGHT_FILES = ("best_ckpt.pth", "osnet_x0_25.pth", "ltc_motion_mot17.pth", "dma_gbm_mot17.gbm")
kaggle_input = Path("/kaggle/input")
cached_weights_dir = None
cached_hf_dir = None
if kaggle_input.is_dir():
    for sub in sorted(kaggle_input.iterdir()):
        if not sub.is_dir():
            continue
        if cached_weights_dir is None and all(any(sub.rglob(w)) for w in WEIGHT_FILES):
            cached_weights_dir = sub
        if cached_hf_dir is None and (any(sub.rglob("*.gguf")) or (sub / "hub").is_dir()):
            cached_hf_dir = sub / "hub" if (sub / "hub").is_dir() else sub

if cached_weights_dir:
    os.environ["EF_WEIGHTS_DIR"] = str(cached_weights_dir)
    print("Smart Cache: reusing weights from", cached_weights_dir)
else:
    os.environ["EF_WEIGHTS_DIR"] = str(GPU_DIR / "weights")
    print("No cached weights in /kaggle/input; will use", os.environ["EF_WEIGHTS_DIR"])

hf_home = Path("/kaggle/working/.hf_home")
hf_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cached_hf_dir.parent if (cached_hf_dir and cached_hf_dir.name == "hub") else hf_home)
if cached_hf_dir:
    print("Smart Cache: detected pre-cached HF/GGUF models at", cached_hf_dir)
print("HF_HOME =", os.environ["HF_HOME"])

In [ ]:
# 3. Install Dependencies, Externals, Weights & Check llama-server Compatibility
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

GPU_DIR = Path("/kaggle/working/ExhibitFlow/GPU_server")
EXT_DIR = GPU_DIR / "external"
EXT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, str(GPU_DIR / "scripts" / "install_deps.py")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "brotli>=1.2"], check=True)

for name, url in (
    ("ByteTrack-DMA-LTC-Motion-Tracker", "https://github.com/khangkaka066/ByteTrack-DMA-LTC-Motion-Tracker.git"),
    ("multicam-2d-floorplan-tracker", "https://github.com/potatips/multicam-2d-floorplan-tracker.git"),
):
    dst = EXT_DIR / name
    if not dst.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dst)], check=True)

weights_dir = Path(os.environ.get("EF_WEIGHTS_DIR") or (GPU_DIR / "weights"))
subprocess.run([sys.executable, str(GPU_DIR / "scripts" / "fetch_weights.py"), str(weights_dir)], check=False)

# Check llama-server availability / glibc compatibility and advise tracking_seg fallback if unavailable
RECOMMENDED_MODE = "full"
llama_rc = subprocess.run(["bash", str(GPU_DIR / "scripts" / "install_llama.sh")], check=False).returncode
if llama_rc != 0:
    RECOMMENDED_MODE = "tracking_seg"
    print("[!] llama-server could not be installed/built on this Kaggle image.")
    print("[!] Automatically advising fallback MODE='tracking_seg' (no VLM required).")
else:
    print("[ok] llama-server check passed. RECOMMENDED_MODE =", RECOMMENDED_MODE)

In [ ]:
# 4. One-Cell GPU Worker Launcher (Set DEMO_MODE = True for pure-CPU test without weights)
import os
import subprocess
import sys
from pathlib import Path

DEMO_MODE = False
PORT = 8001
MODE = globals().get("RECOMMENDED_MODE", "full")
DATA_DIR = "/kaggle/working/gpu_data"

GPU_DIR = Path("/kaggle/working/ExhibitFlow/GPU_server")
cmd = [
    sys.executable,
    "-u",
    str(GPU_DIR / "launch.py"),
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--mode", MODE,
    "--tunnel", "quick",
    "--data-dir", DATA_DIR,
]
if DEMO_MODE:
    cmd.append("--demo")

print("Starting ExhibitFlow GPU Worker:", " ".join(cmd))
subprocess.run(cmd, cwd=str(GPU_DIR), check=True)